# PandaPickCube — обучение в Google Colab (обновлённый стек)

Ноутбук **самодостаточный**: не клонирует репозиторий и не вызывает `train.py` — логика встроена.

## Что изменено по сравнению с оригиналом

Версии пакетов подобраны так, чтобы корректно работать в **Python 3.13** (текущий Colab) и избежать известных конфликтов:

- **numpy 2.5.x** — актуальная ветка, есть wheels для Python 3.13.
- **numba ≥ 0.67** — первая ветка с поддержкой numpy 2.5.
- **JAX `[cuda12]`** — extra называется `cuda12` (не `cuda12-pip`, он удалён с версии 0.6.0).
- **PyTorch `cu128`** — в этом индексе есть wheels для Python 3.13; в `cu124` и старше их нет.
- **NVIDIA-библиотеки зафиксированы** под требования PyTorch 2.11 (`cuDNN 9.19.0.56`, `NCCL 2.28.9`, `NVSHMEM 3.4.5`). Без фиксации JAX подтягивает более новые версии, и pip выдаёт конфликт.
- **`pymc`/`pytensor` удаляются** — они требуют `numba≤0.65.1` и не используются в обучении.

## Перед запуском

1. **Runtime → Change runtime type → GPU** (T4 или лучше).
2. Выполняйте ячейки **сверху вниз**.
3. После установки **обязательно** `Runtime → Restart runtime`, затем ячейка проверки (3) и дальше.

Чекпоинты и логи — папка `logs/` относительно `LOG_ROOT` (по умолчанию `/content`). При монтировании Google Drive задайте `LOG_ROOT` на путь в диске.

## 1. Установка зависимостей

Порядок важен: сначала numpy → numba → JAX → PyTorch с зафиксированными NVIDIA-библиотеками. Так `pip` не подтянет несовместимые версии как транзитивные зависимости.

In [ ]:
# ============================================================
# 1. Базовые научные библиотеки (numpy 2.5 — wheels для Python 3.13)
# ============================================================
%pip install -q --upgrade "numpy>=2.5,<2.6" "scipy"

# 2. numba, совместимая с numpy 2.5 и Python 3.13
%pip install -q --upgrade "numba>=0.67.0"

# 3. JAX с CUDA 12 (extra называется "cuda12")
%pip install -q --upgrade "jax[cuda12]"

# 4. PyTorch cu128 + зафиксированные NVIDIA-библиотеки под его требования
#    (без фиксации JAX подтягивает более новые версии → конфликт)
%pip install -q --upgrade \
    torch torchvision torchaudio \
    nvidia-cudnn-cu12==9.19.0.56 \
    nvidia-nccl-cu12==2.28.9 \
    nvidia-nvshmem-cu12==3.4.5 \
    --index-url https://download.pytorch.org/whl/cu128

# 5. MuJoCo Playground + инструменты
%pip install -q --upgrade "playground[learning]" tensorboard pytest

# 6. Удаляем pymc/pytensor: конфликтуют с numba>=0.67 и не нужны для обучения
%pip uninstall -y -q pymc pytensor

print("\nУстановка завершена. Теперь: Runtime → Restart runtime.")

## 2. Перезапуск и проверка

**Сначала сделайте `Runtime → Restart runtime`** — без этого старые модули останутся в памяти и JAX продолжит видеть `cpu`. Затем выполните ячейку ниже.

In [ ]:
import os
# Убираем LD_LIBRARY_PATH до импорта JAX: Colab иногда выставляет сюда путь,
# конфликтующий с CUDA-библиотеками JAX (причина ошибки CUDA error 303).
os.environ.pop("LD_LIBRARY_PATH", None)

import importlib
for name in ["numpy", "scipy", "numba", "jax", "torch", "tensorboard"]:
    try:
        m = importlib.import_module(name)
        print(f"OK  {name:12s} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"ERR {name:12s} {e}")

import jax, torch
print()
print("JAX backend :", jax.default_backend())
print("JAX devices :", jax.devices())
print("Torch CUDA  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))

assert jax.default_backend() == "gpu", (
    "JAX не видит GPU. Смотрите раздел Troubleshooting в конце ноутбука."
)
assert torch.cuda.is_available(), (
    "PyTorch не видит GPU. Проверьте, что среда выполнения с GPU."
)
print("\nВсё готово к обучению.")

## 3. (Опционально) Google Drive

Раскомментируйте, смонтируйте диск, затем в следующей ячейке задайте `LOG_ROOT` на путь вроде `/content/drive/MyDrive/colab_panda`.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")

## 4. Рабочая папка и переменные окружения

`MUJOCO_GL=egl` — headless-рендер. `XLA_PYTHON_CLIENT_PREALLOCATE=false` — не занимать всю VRAM заранее (важно на T4 с `NUM_ENVS=4096`).

In [ ]:
import os

LOG_ROOT = "/content"  # например "/content/drive/MyDrive/colab_panda"
os.makedirs(LOG_ROOT, exist_ok=True)
os.chdir(LOG_ROOT)

os.environ["MUJOCO_GL"] = "egl"
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.pop("LD_LIBRARY_PATH", None)

print("cwd:", os.getcwd())

## 5. Обучение

Ниже встроены `config.py` и логика из `train.py`. На **T4** при OOM уменьшите `NUM_ENVS` до `1024` или `512`.

In [ ]:
"""Встроенные config.py + train.py для Colab (без внешнего репозитория)."""

import json
import os
import time
from datetime import datetime
from types import SimpleNamespace

import jax
import torch
from mujoco_playground import registry, wrapper_torch
from mujoco_playground.config import manipulation_params
from rsl_rl.runners import OnPolicyRunner

jax.config.update("jax_default_matmul_precision", "highest")

# --- Параметры запуска (аналог аргументов CLI из train.py) ---
NUM_ENVS = 4096
MAX_ITERS = 1001
SAVE_INTERVAL = 50
SEED = 1
EXP_NAME = None  # str или None → имя с датой
RESUME = None  # путь к чекпоинту или None
DEVICE = "cuda:0"

# --- config.py ---
ENV_NAME = "PandaPickCube"

ACTOR_HIDDEN_DIMS = [512, 256, 128]
CRITIC_HIDDEN_DIMS = [512, 256, 128]
ACTIVATION = "elu"
INIT_NOISE_STD = 1.0


def build_runner_cfg(obs_size) -> dict:
    cfg = manipulation_params.rsl_rl_config(ENV_NAME).to_dict()
    _policy = cfg.pop("policy", {})
    cfg["actor"] = {
        "class_name": "rsl_rl.models.mlp_model.MLPModel",
        "hidden_dims": ACTOR_HIDDEN_DIMS,
        "activation": ACTIVATION,
        "distribution_cfg": {
            "class_name": "rsl_rl.modules.distribution.GaussianDistribution",
            "init_std": INIT_NOISE_STD,
        },
    }
    cfg["critic"] = {
        "class_name": "rsl_rl.models.mlp_model.MLPModel",
        "hidden_dims": CRITIC_HIDDEN_DIMS,
        "activation": ACTIVATION,
    }
    if isinstance(obs_size, dict):
        cfg["obs_groups"] = {"actor": ["state"], "critic": ["privileged_state"]}
    else:
        cfg["obs_groups"] = {"actor": ["state"], "critic": ["state"]}
    return cfg


def resolve_log_dir(exp_name, resume) -> str:
    if exp_name:
        name = exp_name
    elif resume:
        name = os.path.basename(os.path.dirname(os.path.abspath(resume)))
    else:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        name = f"{ENV_NAME}_{timestamp}"
    log_dir = os.path.join("logs", name)
    os.makedirs(log_dir, exist_ok=True)
    return log_dir


args = SimpleNamespace(
    exp_name=EXP_NAME,
    num_envs=NUM_ENVS,
    max_iters=MAX_ITERS,
    device=DEVICE,
    resume=RESUME,
    save_interval=SAVE_INTERVAL,
    seed=SEED,
)

device = args.device
device_rank = int(device.split(":")[-1]) if "cuda" in device else 0

log_dir = resolve_log_dir(args.exp_name, args.resume)

print(f"Среда:            {ENV_NAME}")
print(f"Устройство:       {device}")
print(f"Параллельных сред: {args.num_envs}")
print(f"Итерации:         {args.max_iters}")
print(f"Лог/чекпоинты:    {log_dir}")
print()

env_cfg = registry.get_default_config(ENV_NAME)
raw_env = registry.load(ENV_NAME, config=env_cfg)

brax_env = wrapper_torch.RSLRLBraxWrapper(
    raw_env,
    num_actors=args.num_envs,
    seed=args.seed,
    episode_length=env_cfg.episode_length,
    action_repeat=1,
    device_rank=device_rank,
)
brax_env.cfg = {}

cfg_dict = build_runner_cfg(raw_env.observation_size)
cfg_dict["seed"] = args.seed
cfg_dict["max_iterations"] = args.max_iters
cfg_dict["save_interval"] = args.save_interval

config_path = os.path.join(log_dir, "config.json")
with open(config_path, "w") as f:
    json.dump(cfg_dict, f, indent=2, default=str)
print(f"Конфиг сохранён: {config_path}")

runner = OnPolicyRunner(brax_env, cfg_dict, log_dir, device=device)

if args.resume:
    print(f"Загрузка чекпоинта: {args.resume}")
    runner.load(args.resume)

print("=" * 60)
print("  Начинаю обучение...")
print("=" * 60)
print()

t_start = time.time()
runner.learn(
    num_learning_iterations=args.max_iters,
    init_at_random_ep_len=False,
)
elapsed = time.time() - t_start

print()
print("=" * 60)
print(f"  Обучение завершено за {elapsed:.0f} с ({elapsed / 60:.1f} мин)")
print(f"  Логи и чекпоинты: {log_dir}")
print("=" * 60)

## 6. (Опционально) TensorBoard

Логи: `logs/<имя_эксперимента>/` относительно `LOG_ROOT`.

In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir logs

## Troubleshooting

### `JAX backend: cpu` / `RuntimeError: Unknown CUDA error 303`

JAX не смог загрузить CUDA-библиотеки. Пошагово:

1. Убедитесь, что `Runtime → Restart runtime` был выполнен после установки.
2. В ячейке проверки `os.environ.pop("LD_LIBRARY_PATH", None)` стоит **до** `import jax` — если нет, добавьте.
3. Переустановите JAX принудительно с явным индексом:
   ```python
   %pip uninstall -y jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt
   %pip install -q --force-reinstall "jax[cuda12]" \
       -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
   ```
   Затем `Runtime → Restart runtime` и снова ячейку проверки.
4. Проверьте драйвер: `!nvidia-smi`. Если драйвер старше 525, JAX с CUDA 12 не запустится — смените тип GPU или среду выполнения.

### pip ругается: `torch requires nvidia-cudnn-cu12==X.Y.Z`

JAX подтянул более новые NVIDIA-библиотеки, чем ожидает PyTorch. Повторите установку PyTorch с явной фиксацией (ячейка 1, шаг 4). Если не помогает — добавьте `--force-reinstall`:

```python
%pip install -q --force-reinstall --no-deps \
    nvidia-cudnn-cu12==9.19.0.56 \
    nvidia-nccl-cu12==2.28.9 \
    nvidia-nvshmem-cu12==3.4.5 \
    --index-url https://download.pytorch.org/whl/cu128
```

### pip ругается: `pymc requires pytensor`

`pymc` не нужен для обучения, просто удалите его (шаг 6 ячейки 1). Если после этого импорт `pymc` нужен в другом коде — установите `pytensor==2.38.3`, но учтите, что он требует `numba<=0.68.0`, что конфликтует с текущим стеком.

### OOM на T4

Уменьшите `NUM_ENVS` (до `1024` или `512`) в ячейке 5. Также помогает `os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.9"` перед импортом JAX.